# 06 - 方案 B 快速验证：线性变形层

**假设**：C-MSTNF 的高频跳变源于 5 层 ReLU 变形 MLP。如果用单层线性映射替代，变形场天然 Lipschitz 连续，跳变应消失。

**验证方式**：
1. 在同一 canonical field 上分别训练 MLP 变形和线性变形
2. 对比渲染质量、3D 场光滑度、时序连续性
3. 如果线性变形已能收敛（虽然可能精度稍低），说明 MLP 不是必要的

In [1]:
import sys, os, glob, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

%matplotlib inline

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

CUDA_DEVICE = 0
os.environ['CUDA_VISIBLE_DEVICES'] = str(CUDA_DEVICE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 1. 定义线性变形模型

对比两种变形场：
- **MLP 变形**（原始）：5 层 ReLU MLP，`pos_enc + state + action → (Δx, Δy, Δz)`
- **线性变形**（实验）：单层线性映射 + 低频位置编码，`pos_enc + state + action → (Δx, Δy, Δz)`

In [2]:
from src.models.layers import PositionalEncoder, MLPDecoder
from src.models.model_mstnf import MultiScaleEMA
from src.models.model_cmstnf import CanonicalField


class LinearDeformModel(nn.Module):
    """线性变形 C-MSTNF：变形场仅用单层线性映射。

    与 CMSTNFModel 接口一致，唯一区别是 deform_mlp 被替换为 nn.Linear。
    """

    def __init__(self, action_dim, window_size=20, n_scales=4, hidden_dim=128,
                 d_filter=128, n_freqs=10, deform_n_freqs=4):
        super().__init__()
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim

        self.canonical = CanonicalField(d_filter=d_filter, n_freqs=n_freqs)

        # 时序编码器（与 C-MSTNF 完全一致）
        self.temporal = MultiScaleEMA(
            action_dim=action_dim, n_scales=n_scales,
            window_size=window_size, hidden_dim=hidden_dim,
        )

        # 低频位置编码（比原始更低，进一步保证光滑）
        self.deform_encoder = PositionalEncoder(d_input=3, n_freqs=deform_n_freqs, log_space=True)
        deform_enc_dim = 3 * (1 + 2 * deform_n_freqs)

        # 核心改动：单层线性映射替代 5 层 MLP
        self.deform_linear = nn.Linear(deform_enc_dim + hidden_dim + action_dim, 3)

        # 位移初始化为接近零
        with torch.no_grad():
            self.deform_linear.bias.zero_()
            nn.init.xavier_uniform_(self.deform_linear.weight, gain=0.01)

    def _compute_displacement(self, points, action_window):
        B, K, D = action_window.shape
        physics_state = self.temporal(action_window)
        current_action = action_window[:, -1, :]

        N_rays = points.shape[0]
        n_samples = points.shape[1]

        pts_exp = points.unsqueeze(0).expand(B, -1, -1, -1).reshape(B * N_rays, n_samples, 3)
        x_deform = self.deform_encoder(pts_exp).reshape(-1, self.deform_encoder.d_output)

        state_exp = physics_state.unsqueeze(1).expand(-1, N_rays, -1).reshape(B * N_rays, self.hidden_dim)
        state_flat = state_exp.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, self.hidden_dim)

        action_exp = current_action.unsqueeze(1).expand(-1, N_rays, -1).reshape(B * N_rays, D)
        action_flat = action_exp.unsqueeze(1).expand(-1, n_samples, -1).reshape(-1, D)

        latent = torch.cat([x_deform, state_flat, action_flat], dim=-1)
        displacement = self.deform_linear(latent).reshape(B * N_rays, n_samples, 3)
        return displacement, physics_state

    # 接口与 CMSTNFModel 一致
    def forward_canonical(self, points):
        return self.canonical(points)

    def forward(self, points, action_window):
        B = action_window.shape[0]
        N_rays = points.shape[0]
        n_samples = points.shape[1]
        displacement, _ = self._compute_displacement(points, action_window)
        pts_exp = points.unsqueeze(0).expand(B, -1, -1, -1).reshape(B * N_rays, n_samples, 3)
        return self.canonical(pts_exp + displacement)

    def compute_smoothness(self, aw_t, aw_t1):
        s_t = self.temporal(aw_t)
        s_t1 = self.temporal(aw_t1)
        return torch.mean((s_t1 - s_t) ** 2)

    def freeze_canonical(self):
        for p in self.canonical.parameters():
            p.requires_grad = False

    def unfreeze_canonical(self):
        for p in self.canonical.parameters():
            p.requires_grad = True

    def get_learned_decays(self):
        return self.temporal.decays.detach().cpu().numpy()

print('LinearDeformModel defined.')
print(f'  deform params: Linear(d_in -> 3) vs MLP(5 layers, d_filter=128)')
print(f'  positional encoding: deform_n_freqs=4 (vs 6 in C-MSTNF)')

LinearDeformModel defined.
  deform params: Linear(d_in -> 3) vs MLP(5 layers, d_filter=128)
  positional encoding: deform_n_freqs=4 (vs 6 in C-MSTNF)


## 2. 加载数据和训练基础设施

In [3]:
from src.data.dataset import SoftSequenceDataset
from src.utils.camera import get_rays
from src.utils.rendering import OM_rendering, sample_stratified
from src.config.params import load_config, get_camera_params

# 数据
CANON_DATA = os.path.join(PROJECT_ROOT, 'data', 'canonical_data')
SEQ_DATA = os.path.join(PROJECT_ROOT, 'data', 'sequence_data')

train_cfg = load_config('training')
cam_cfg = get_camera_params()

NEAR, FAR = cam_cfg['near'], cam_cfg['far']
N_SAMPLES = cam_cfg['n_samples']
WINDOW = train_cfg['temporal']['window_size']

print(f'Config: near={NEAR}, far={FAR}, n_samples={N_SAMPLES}, window={WINDOW}')

# Canonical 数据
canon_files = sorted(glob.glob(os.path.join(CANON_DATA, '*.npz')))
canon_ds = SoftSequenceDataset(CANON_DATA, seq_len=1, file_list=canon_files)
print(f'Canonical: {len(canon_ds)} samples, img={canon_ds.H}x{canon_ds.W}, action_dim={canon_ds.action_dim}')

# Sequence 数据
seq_files = sorted(glob.glob(os.path.join(SEQ_DATA, '*.npz')))
split = max(1, int(0.8 * len(seq_files)))
train_files, val_files = seq_files[:split], seq_files[split:]

train_ds = SoftSequenceDataset(SEQ_DATA, seq_len=WINDOW, file_list=train_files, return_pairs=True)
val_ds = SoftSequenceDataset(SEQ_DATA, seq_len=WINDOW, file_list=val_files, norm_factor=train_ds.norm_factor)
action_dim = train_ds.action_dim
print(f'Sequence: train={len(train_ds)}, val={len(val_ds)}, action_dim={action_dim}')

# 相机
focal = torch.tensor(float(train_ds.focal), device=device)
rays_o, rays_d = get_rays(train_ds.H, train_ds.W, focal,
                          cam_cfg['eye'], cam_cfg['center'], cam_cfg['up'])
rays_o = rays_o.to(device)
rays_d = rays_d.to(device)
H, W = train_ds.H, train_ds.W
print(f'Camera: {H}x{W}, focal={train_ds.focal:.1f}')

Config: near=0.5, far=2.5, n_samples=64, window=20
Norm Factor: 1.0
Canonical: 200 samples, img=100x100, action_dim=2
Norm Factor: 0.005
Sequence: train=1497, val=500, action_dim=2
Camera: 100x100, focal=130.0


## 3. Phase 1：训练共享 Canonical Field

两种模型共享同一个 canonical field，排除 canonical 差异的影响。

In [4]:
PHASE1_EPOCHS = 30  # 快速验证用较少 epoch

canon_model = CanonicalField(d_filter=128, n_freqs=10).to(device)
optimizer_c = torch.optim.Adam(canon_model.parameters(), lr=1e-3)
loader_c = DataLoader(canon_ds, batch_size=4, shuffle=True, num_workers=2)

print(f'Phase 1: Training canonical field ({PHASE1_EPOCHS} epochs)...')
for epoch in range(1, PHASE1_EPOCHS + 1):
    canon_model.train()
    epoch_loss = 0
    for batch in loader_c:
        _, img = batch[0], batch[1]
        img = img.to(device)
        B = img.shape[0]

        # 前景过采样
        fg_mask = img[0] > 0.1
        fg_idx = torch.where(fg_mask)[0]
        n_fg, n_bg = 512, 512
        if len(fg_idx) > 0:
            sel_fg = fg_idx[torch.randint(len(fg_idx), (n_fg,), device=device)]
            sel_bg = torch.randint(H * W, (n_bg,), device=device)
            sel = torch.cat([sel_fg, sel_bg])
        else:
            sel = torch.randint(H * W, (n_fg + n_bg,), device=device)

        pts, _ = sample_stratified(rays_o[sel], rays_d[sel], NEAR, FAR, N_SAMPLES)

        # 分块渲染
        raw_parts = []
        for i in range(0, pts.shape[0], 4096):
            raw_parts.append(canon_model(pts[i:i+4096]))
        raw = torch.cat(raw_parts, dim=0)
        rgb_map, _ = OM_rendering(raw)
        pred = rgb_map.unsqueeze(0).expand(B, -1)
        gt = img[:, sel]
        loss = F.mse_loss(pred, gt)

        optimizer_c.zero_grad()
        loss.backward()
        optimizer_c.step()
        epoch_loss += loss.item()

    if epoch % 10 == 0 or epoch == PHASE1_EPOCHS:
        avg = epoch_loss / len(loader_c)
        print(f'  Epoch {epoch}/{PHASE1_EPOCHS} | Loss: {avg:.5f}')

print('Phase 1 done!')

# 保存 canonical state dict
canon_state = copy.deepcopy(canon_model.state_dict())

Phase 1: Training canonical field (30 epochs)...
  Epoch 10/30 | Loss: 0.00244
  Epoch 20/30 | Loss: 0.00135
  Epoch 30/30 | Loss: 0.00042
Phase 1 done!


## 4. Phase 2：分别训练 MLP 变形 vs 线性变形

使用相同的 canonical 权重、相同的数据、相同的超参数（除了变形网络结构）。

In [5]:
PHASE2_EPOCHS = 60  # 快速验证
LR = 5e-4
BATCH_SIZE = 4

from src.models.model_cmstnf import CMSTNFModel

# 创建两个模型，加载相同 canonical
model_mlp = CMSTNFModel(action_dim=action_dim, window_size=WINDOW, hidden_dim=128).to(device)
model_linear = LinearDeformModel(action_dim=action_dim, window_size=WINDOW, hidden_dim=128).to(device)

model_mlp.canonical.load_state_dict(canon_state)
model_linear.canonical.load_state_dict(canon_state)
model_mlp.freeze_canonical()
model_linear.freeze_canonical()

# 参数量对比
def count_params(model, name):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    deform = sum(p.numel() for n, p in model.named_parameters() if 'deform' in n and p.requires_grad)
    print(f'{name}: total={total:,}, trainable={trainable:,}, deform={deform:,}')

count_params(model_mlp, 'MLP Deform')
count_params(model_linear, 'Linear Deform')

# 数据加载器
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

w_recon, w_next, w_smooth = 0.7, 0.3, 0.1

MLP Deform: total=292,297, trainable=168,839, deform=168,839
Linear Deform: total=142,112, trainable=18,654, deform=474


In [1]:
def train_phase2(model, name, n_epochs=PHASE2_EPOCHS):
    """训练 Phase 2 并返回每 epoch 的 loss 记录。"""
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=LR)

    history = {'recon': [], 'smooth': [], 'total': []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0
        epoch_recon = 0
        epoch_smooth = 0

        pbar = tqdm(train_loader, desc=f'[{name}] Epoch {epoch}/{n_epochs}', leave=False)
        for seq_t, seq_t1, img_t, img_t1 in pbar:
            seq_t = seq_t.to(device)
            seq_t1 = seq_t1.to(device)
            img_t = img_t.to(device)
            B = img_t.shape[0]

            # 前景过采样
            fg_mask = img_t[0] > 0.1
            fg_idx = torch.where(fg_mask)[0]
            n_fg, n_bg = 512, 512
            if len(fg_idx) > 0:
                sel_fg = fg_idx[torch.randint(len(fg_idx), (n_fg,), device=device)]
                sel_bg = torch.randint(H * W, (n_bg,), device=device)
                sel = torch.cat([sel_fg, sel_bg])
            else:
                sel = torch.randint(H * W, (n_fg + n_bg,), device=device)

            pts, _ = sample_stratified(rays_o[sel], rays_d[sel], NEAR, FAR, N_SAMPLES)
            N_rays = pts.shape[0]

            # 重建当前帧
            # model(pts, seq_t) 内部按 B 扩展: 输出 (B*N_rays, n_samples, 2)
            raw_parts = []
            for i in range(0, N_rays, 4096):
                raw_parts.append(model(pts[i:i+4096], seq_t))
            raw = torch.cat(raw_parts, dim=0).reshape(B * N_rays, N_SAMPLES, 2)
            rgb_map, _ = OM_rendering(raw)
            pred_t = rgb_map.reshape(B, N_rays)
            loss_recon = F.mse_loss(pred_t, img_t[:, sel])

            # 重建下一帧
            raw_parts2 = []
            for i in range(0, N_rays, 4096):
                raw_parts2.append(model(pts[i:i+4096], seq_t1))
            raw2 = torch.cat(raw_parts2, dim=0).reshape(B * N_rays, N_SAMPLES, 2)
            rgb_map2, _ = OM_rendering(raw2)
            pred_t1 = rgb_map2.reshape(B, N_rays)
            loss_next = F.mse_loss(pred_t1, img_t1[:, sel].to(device))

            # 平滑
            loss_smooth = model.compute_smoothness(seq_t, seq_t1)

            loss = w_recon * loss_recon + w_next * loss_next + w_smooth * loss_smooth

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_recon += loss_recon.item()
            epoch_smooth += loss_smooth.item()
            pbar.set_postfix({'loss': f'{loss.item():.5f}'})

        n = max(len(train_loader), 1)
        history['total'].append(epoch_loss / n)
        history['recon'].append(epoch_recon / n)
        history['smooth'].append(epoch_smooth / n)

        if epoch % 20 == 0 or epoch == n_epochs:
            print(f'[{name}] Epoch {epoch} | Total: {history["total"][-1]:.5f} | '
                  f'Recon: {history["recon"][-1]:.5f} | Smooth: {history["smooth"][-1]:.6f}')

    return history

print('Training function defined.')

NameError: name 'PHASE2_EPOCHS' is not defined

In [7]:
print('='*60)
print('Training MLP Deform (baseline)...')
print('='*60)
hist_mlp = train_phase2(model_mlp, 'MLP')

Training MLP Deform (baseline)...


[MLP] Epoch 1/60:   0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_209429/3135757154.py:43: UserWarning: Using a target size (torch.Size([4, 1024])) that is different to the input size (torch.Size([4, 4096])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss_recon = F.mse_loss(pred_t, img_t[:, sel])


RuntimeError: The size of tensor a (4096) must match the size of tensor b (1024) at non-singleton dimension 1

In [ ]:
print('='*60)
print('Training Linear Deform (experiment)...')
print('='*60)
hist_linear = train_phase2(model_linear, 'Linear')

## 5. 训练曲线对比

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = range(1, PHASE2_EPOCHS + 1)

for ax, key, title in zip(axes, ['total', 'recon', 'smooth'],
                          ['Total Loss', 'Reconstruction Loss', 'Smoothness Loss']):
    ax.plot(epochs, hist_mlp[key], 'b-', label='MLP Deform', linewidth=2)
    ax.plot(epochs, hist_linear[key], 'r--', label='Linear Deform', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Loss: MLP vs Linear Deformation', fontsize=14)
plt.tight_layout()
plt.show()

## 6. 渲染质量对比

在验证集上对比两模型的渲染图像与 GT。

In [ ]:
def render_full(model, val_seq):
    """完整渲染一帧。"""
    model.eval()
    with torch.no_grad():
        pts, _ = sample_stratified(rays_o, rays_d, NEAR, FAR, N_SAMPLES, perturb=False)
        N_rays = pts.shape[0]

        B = val_seq.shape[0]
        raw_parts = []
        for i in range(0, N_rays, 4096):
            chunk = pts[i:i+4096]
            raw_parts.append(model(chunk, val_seq))
        raw = torch.cat(raw_parts, dim=0).reshape(N_rays, N_SAMPLES, 2)
        rgb_map, _ = OM_rendering(raw)
        return rgb_map.reshape(H, W).cpu().numpy()

# 选几个验证帧
frame_indices = [0, len(val_ds)//4, len(val_ds)//2, 3*len(val_ds)//4, len(val_ds)-1]
frame_indices = [min(i, len(val_ds)-1) for i in frame_indices]

fig, axes = plt.subplots(3, len(frame_indices), figsize=(4*len(frame_indices), 12))
row_labels = ['GT', 'MLP Deform', 'Linear Deform']

for col, fi in enumerate(frame_indices):
    val_seq, val_img = val_ds[fi]
    val_seq = val_seq.unsqueeze(0).to(device)
    gt = val_img.reshape(H, W).numpy()

    pred_mlp = render_full(model_mlp, val_seq)
    pred_linear = render_full(model_linear, val_seq)

    axes[0, col].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'Frame {fi}')
    axes[1, col].imshow(pred_mlp, cmap='gray', vmin=0, vmax=1)
    mse_mlp = np.mean((pred_mlp - gt)**2)
    axes[1, col].set_title(f'MSE={mse_mlp:.5f}')
    axes[2, col].imshow(pred_linear, cmap='gray', vmin=0, vmax=1)
    mse_lin = np.mean((pred_linear - gt)**2)
    axes[2, col].set_title(f'MSE={mse_lin:.5f}')

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=12)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Rendering Quality: MLP vs Linear Deformation', fontsize=14)
plt.tight_layout()
plt.show()

## 7. 3D 变形场光滑度分析

核心验证：线性变形场是否真的更光滑？

指标：
- 变形场在空间上的 Jacobian 范数（衡量空间光滑度）
- 相邻帧变形差异（衡量时序连续性）

In [ ]:
# 3D 网格查询变形场
GRID_RES = 20
xs = np.linspace(-0.1, 0.1, GRID_RES)
ys = np.linspace(-0.1, 0.1, GRID_RES)
zs = np.linspace(0.0, 0.55, GRID_RES)
gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
grid_pts = np.stack([gx.flatten(), gy.flatten(), gz.flatten()], axis=-1)
grid_tensor = torch.tensor(grid_pts, dtype=torch.float32, device=device)

# 选一帧的动作
val_seq, _ = val_ds[len(val_ds)//2]
val_seq = val_seq.unsqueeze(0).to(device)

print('Computing displacement fields...')

with torch.no_grad():
    pts_grid = grid_tensor.unsqueeze(1)  # (N, 1, 3)

    # MLP 变形（CMSTNFModel.deform 接口: points(N_rays, n_samples, 3), action_window(B,K,D)）
    disp_mlp_raw = []
    for i in range(0, len(pts_grid), 4096):
        chunk = pts_grid[i:i+4096]
        d, _ = model_mlp.deform(chunk, val_seq)
        disp_mlp_raw.append(d)
    disp_mlp = torch.cat(disp_mlp_raw, dim=0).squeeze(1).cpu().numpy()  # (N, 3)

    # 线性变形
    disp_lin_raw = []
    for i in range(0, len(pts_grid), 4096):
        chunk = pts_grid[i:i+4096]
        d, _ = model_linear._compute_displacement(chunk, val_seq)
        disp_lin_raw.append(d)
    disp_lin = torch.cat(disp_lin_raw, dim=0).squeeze(1).cpu().numpy()  # (N, 3)

print(f'MLP displacement:     range [{disp_mlp.min():.6f}, {disp_mlp.max():.6f}], '
      f'std={disp_mlp.std():.6f}')
print(f'Linear displacement:  range [{disp_lin.min():.6f}, {disp_lin.max():.6f}], '
      f'std={disp_lin.std():.6f}')

In [ ]:
# 空间光滑度：计算 Jacobian 近似（沿 z 轴的有限差分）
def spatial_smoothness(disp, grid_res):
    """计算变形场沿 z 轴的空间导数范数。"""
    d = disp.reshape(grid_res, grid_res, grid_res, 3)
    # 沿 z 方向差分
    dz = np.diff(d, axis=2, n=1)
    dz_norm = np.linalg.norm(dz, axis=-1)
    return dz_norm.mean(), dz_norm.std(), dz_norm

mean_mlp, std_mlp, grad_z_mlp = spatial_smoothness(disp_mlp, GRID_RES)
mean_lin, std_lin, grad_z_lin = spatial_smoothness(disp_lin, GRID_RES)

print(f'=== Spatial Smoothness (z-gradient) ===')
print(f'MLP:    mean={mean_mlp:.6f}, std={std_mlp:.6f}, max={grad_z_mlp.max():.6f}')
print(f'Linear: mean={mean_lin:.6f}, std={std_lin:.6f}, max={grad_z_lin.max():.6f}')

# XZ 切片可视化（y=中间）
mid_y = GRID_RES // 2
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 位移大小
disp_mlp_mag = np.linalg.norm(disp_mlp, axis=-1).reshape(GRID_RES, GRID_RES, GRID_RES)[:, mid_y, :]
disp_lin_mag = np.linalg.norm(disp_lin, axis=-1).reshape(GRID_RES, GRID_RES, GRID_RES)[:, mid_y, :]

im0 = axes[0, 0].imshow(disp_mlp_mag.T, extent=[xs[0], xs[-1], zs[0], zs[-1]],
                         origin='lower', cmap='viridis', aspect='auto')
axes[0, 0].set_title('MLP |Displacement| (XZ)')
axes[0, 0].set_xlabel('X'); axes[0, 0].set_ylabel('Z')
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(disp_lin_mag.T, extent=[xs[0], xs[-1], zs[0], zs[-1]],
                         origin='lower', cmap='viridis', aspect='auto')
axes[0, 1].set_title('Linear |Displacement| (XZ)')
axes[0, 1].set_xlabel('X'); axes[0, 1].set_ylabel('Z')
plt.colorbar(im1, ax=axes[0, 1])

# z 方向梯度
grad_mlp_xz = grad_z_mlp[:, mid_y, :]
grad_lin_xz = grad_z_lin[:, mid_y, :]

im2 = axes[1, 0].imshow(grad_mlp_xz.T, extent=[xs[0], xs[-1], zs[0], zs[-1]],
                         origin='lower', cmap='hot', aspect='auto')
axes[1, 0].set_title('MLP z-gradient (smoothness)')
axes[1, 0].set_xlabel('X'); axes[1, 0].set_ylabel('Z')
plt.colorbar(im2, ax=axes[1, 0])

im3 = axes[1, 1].imshow(grad_lin_xz.T, extent=[xs[0], xs[-1], zs[0], zs[-1]],
                         origin='lower', cmap='hot', aspect='auto')
axes[1, 1].set_title('Linear z-gradient (smoothness)')
axes[1, 1].set_xlabel('X'); axes[1, 1].set_ylabel('Z')
plt.colorbar(im3, ax=axes[1, 1])

plt.suptitle('Deformation Field Comparison (XZ slice)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. 时序连续性对比

沿一个序列逐步预测，看相邻帧之间的渲染跳变程度。

In [ ]:
# 时序连续性测试：逐步推进 action window
n_frames = min(50, len(val_ds))
step = max(1, len(val_ds) // n_frames)
frame_ids = list(range(0, len(val_ds), step))[:n_frames]

preds_mlp = []
preds_linear = []
gts = []

model_mlp.eval()
model_linear.eval()

with torch.no_grad():
    for fi in tqdm(frame_ids, desc='Rendering sequence'):
        seq, img = val_ds[fi]
        seq = seq.unsqueeze(0).to(device)
        gt = img.reshape(H, W).numpy()
        gts.append(gt)
        preds_mlp.append(render_full(model_mlp, seq))
        preds_linear.append(render_full(model_linear, seq))

# 计算相邻帧差异
diff_mlp = [np.mean((preds_mlp[i+1] - preds_mlp[i])**2) for i in range(len(preds_mlp)-1)]
diff_linear = [np.mean((preds_linear[i+1] - preds_linear[i])**2) for i in range(len(preds_linear)-1)]
diff_gt = [np.mean((gts[i+1] - gts[i])**2) for i in range(len(gts)-1)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(diff_gt, 'g-', alpha=0.5, label='GT inter-frame', linewidth=1)
ax1.plot(diff_mlp, 'b-', label='MLP inter-frame', linewidth=2)
ax1.plot(diff_linear, 'r--', label='Linear inter-frame', linewidth=2)
ax1.set_xlabel('Frame')
ax1.set_ylabel('MSE to next frame')
ax1.set_title('Temporal Continuity (lower = smoother)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 每帧的 MSE
mse_mlp = [np.mean((p - g)**2) for p, g in zip(preds_mlp, gts)]
mse_linear = [np.mean((p - g)**2) for p, g in zip(preds_linear, gts)]
ax2.plot(mse_mlp, 'b-', label='MLP vs GT', linewidth=2)
ax2.plot(mse_linear, 'r--', label='Linear vs GT', linewidth=2)
ax2.set_xlabel('Frame')
ax2.set_ylabel('MSE')
ax2.set_title('Per-frame Reconstruction Quality')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'MLP avg MSE:    {np.mean(mse_mlp):.6f}')
print(f'Linear avg MSE: {np.mean(mse_linear):.6f}')
print(f'MLP inter-frame:    {np.mean(diff_mlp):.6f}')
print(f'Linear inter-frame: {np.mean(diff_linear):.6f}')

## 9. 结论

| 指标 | MLP Deform | Linear Deform | 判定 |
|------|-----------|---------------|------|
| 训练 loss 收敛 | | | |
| 渲染 MSE | | | |
| 空间光滑度 | | | |
| 时序连续性 | | | |
| 参数量 | | | |

**预期结果**：
- 如果 Linear 的渲染质量接近 MLP 且更光滑 → MLP 确实是高频跳变的根源，应去掉
- 如果 Linear 质量明显差于 MLP → MLP 的非线性是必要的，需要其他方法解决高频问题
- 如果两者质量都差 → 问题不在变形网络，可能在时序编码或数据本身